In [32]:
import pandas as pd
import numpy as np
# Carga del dataset transaccional

# Cargar el dataset de órdenes
df_ordenes = pd.read_csv('Raw/ventas_order.csv')

# Inspección visual de la "suciedad" de los textos
print("--- Muestra de Órdenes (Cabeceras) ---")
display(df_ordenes.head())

print("\n--- Información Estructural ---")
df_ordenes.info()

--- Muestra de Órdenes (Cabeceras) ---


,id_orden,fecha_orden,id_cliente,vendedor_region,metodo_pago,nombre_cajero,nro_sesion,canal_venta,id_terminal,turno
0,1022,13-05-2023,2057,Lucas M. – GBA Norte,efectivo,jfernandez,Ses181407,Tel.,TRM-680,Mañana
1,1099,21/12/2022,2038,Sofía R. – CABA,efectivo,mlopez,SESSION729282,Tel.,TRM-423,noche
2,1062,18-12-2024,2030,Valeria T. – Córdoba,TARJETA DE CRÉDITO,jfernandez,ses771157,online,TRM-970,tarde
3,1399,31-05-2022,2002,Camila H. – Mendoza,efectivo,mlopez,ses681320,presencial,TRM-893,TARDE
4,1128,15/11/2023,2018,Diego F. – Rosario,transferencia,csanchez,Ses613990,ONLINE,TRM-793,tarde



--- Información Estructural ---
<class 'pandas.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id_orden         440 non-null    int64
 1   fecha_orden      440 non-null    str  
 2   id_cliente       440 non-null    int64
 3   vendedor_region  440 non-null    str  
 4   metodo_pago      440 non-null    str  
 5   nombre_cajero    440 non-null    str  
 6   nro_sesion       440 non-null    str  
 7   canal_venta      440 non-null    str  
 8   id_terminal      440 non-null    str  
 9   turno            440 non-null    str  
dtypes: int64(2), str(8)
memory usage: 34.5 KB


In [33]:
# 1. Transformación de IDs a texto (para proteger integridad)
df_ordenes['id_orden'] = df_ordenes['id_orden'].astype(str)
df_ordenes['id_cliente'] = df_ordenes['id_cliente'].astype(str)

# 2. Parseo de fechas (Controlando formatos mixtos como 13-05-2023 y 21/12/2022)
#df_ordenes['fecha_orden'] = pd.to_datetime(df_ordenes['fecha_orden'], dayfirst=True, errors='coerce')

# 3. Auditoría y purgado de duplicados en la clave primaria (id_orden)
duplicados_orden = df_ordenes.duplicated(subset=['id_orden']).sum()
print(f"Alerta de integridad: Se encontraron {duplicados_orden} IDs de orden duplicadas.")

if duplicados_orden > 0:
    # Nos quedamos con la primera aparición para no duplicar tickets
    df_ordenes = df_ordenes.drop_duplicates(subset=['id_orden'], keep='first')
    print("Las cabeceras duplicadas han sido eliminadas.")

Alerta de integridad: Se encontraron 20 IDs de orden duplicadas.
Las cabeceras duplicadas han sido eliminadas.


In [34]:

# 2. Estandarización de Métodos de Pago y Turnos (y solución de tildes)
df_ordenes['metodo_pago'] = df_ordenes['metodo_pago'].astype(str).str.title().str.strip()
# Reemplazamos la "é" por "e" para unificar las tarjetas de crédito
df_ordenes['metodo_pago'] = df_ordenes['metodo_pago'].str.replace('é', 'e', regex=False)

df_ordenes['turno'] = df_ordenes['turno'].astype(str).str.title().str.strip()

# 3. Estandarización de Canal de Venta
df_ordenes['canal_venta'] = df_ordenes['canal_venta'].astype(str).str.title().str.replace('.', '', regex=False).str.strip()

# 4. Separación de Vendedor y Región
split_vendedor = df_ordenes['vendedor_region'].str.split('–', expand=True)
df_ordenes['vendedor'] = split_vendedor[0].str.strip()
df_ordenes['region_venta'] = split_vendedor[1].str.strip()
df_ordenes = df_ordenes.drop(columns=['vendedor_region'])

# 5. Estandarización Avanzada de Sesiones (Unificación de prefijos) y Terminales
df_ordenes['nro_sesion'] = df_ordenes['nro_sesion'].astype(str).str.upper().str.strip()
# Ahora todas las sesiones que decían "SESSION" dirán "SES"
df_ordenes['nro_sesion'] = df_ordenes['nro_sesion'].str.replace('SESSION', 'SES', regex=False)

df_ordenes['id_terminal'] = df_ordenes['id_terminal'].astype(str).str.upper().str.strip()

# Verificamos los resultados de la corrección
print("--- Verificación de Datos Normalizados ---")
display(df_ordenes[['fecha_orden', 'metodo_pago', 'nro_sesion']].head())
print("\nValores únicos de Método de Pago:", df_ordenes['metodo_pago'].unique())

--- Verificación de Datos Normalizados ---


,fecha_orden,metodo_pago,nro_sesion
0,13-05-2023,Efectivo,SES181407
1,21/12/2022,Efectivo,SES729282
2,18-12-2024,Tarjeta De Credito,SES771157
3,31-05-2022,Efectivo,SES681320
4,15/11/2023,Transferencia,SES613990



Valores únicos de Método de Pago: <StringArray>
['Efectivo', 'Tarjeta De Credito', 'Transferencia', 'Cheque']
Length: 4, dtype: str


In [35]:
df_ordenes

,id_orden,fecha_orden,id_cliente,metodo_pago,nombre_cajero,nro_sesion,canal_venta,id_terminal,turno,vendedor,region_venta
0,1022,13-05-2023,2057,Efectivo,jfernandez,SES181407,Tel,TRM-680,Mañana,Lucas M.,GBA Norte
1,1099,21/12/2022,2038,Efectivo,mlopez,SES729282,Tel,TRM-423,Noche,Sofía R.,CABA
2,1062,18-12-2024,2030,Tarjeta De Credito,jfernandez,SES771157,Online,TRM-970,Tarde,Valeria T.,Córdoba
3,1399,31-05-2022,2002,Efectivo,mlopez,SES681320,Presencial,TRM-893,Tarde,Camila H.,Mendoza
4,1128,15/11/2023,2018,Transferencia,csanchez,SES613990,Online,TRM-793,Tarde,Diego F.,Rosario
...,...,...,...,...,...,...,...,...,...,...,...
435,1239,10-11-2024,2036,Tarjeta De Credito,jfernandez,SES108800,Tel,TRM-184,Noche,Ana P.,CABA
436,1290,2022-12-03,2076,Tarjeta De Credito,csanchez,SES105459,Tel,TRM-512,Mañana,Camila H.,Mendoza
437,1115,13-07-2024,2074,Transferencia,lrodriguez,SES562458,Tel,TRM-893,Tarde,Valeria T.,Córdoba
438,1278,22-03-2022,2060,Cheque,csanchez,SES512731,Presencial,TRM-970,Mañana,Diego F.,Rosario


In [36]:
df_ordenes['fecha_orden'] = pd.to_datetime(
    df_ordenes['fecha_orden'],
    format='mixed',
    dayfirst=True
)

print(df_ordenes['fecha_orden'].head())

0   2023-05-13
1   2022-12-21
2   2024-12-18
3   2022-05-31
4   2023-11-15
Name: fecha_orden, dtype: datetime64[us]


In [37]:
df_ordenes

,id_orden,fecha_orden,id_cliente,metodo_pago,nombre_cajero,nro_sesion,canal_venta,id_terminal,turno,vendedor,region_venta
0,1022,2023-05-13,2057,Efectivo,jfernandez,SES181407,Tel,TRM-680,Mañana,Lucas M.,GBA Norte
1,1099,2022-12-21,2038,Efectivo,mlopez,SES729282,Tel,TRM-423,Noche,Sofía R.,CABA
2,1062,2024-12-18,2030,Tarjeta De Credito,jfernandez,SES771157,Online,TRM-970,Tarde,Valeria T.,Córdoba
3,1399,2022-05-31,2002,Efectivo,mlopez,SES681320,Presencial,TRM-893,Tarde,Camila H.,Mendoza
4,1128,2023-11-15,2018,Transferencia,csanchez,SES613990,Online,TRM-793,Tarde,Diego F.,Rosario
...,...,...,...,...,...,...,...,...,...,...,...
435,1239,2024-11-10,2036,Tarjeta De Credito,jfernandez,SES108800,Tel,TRM-184,Noche,Ana P.,CABA
436,1290,2022-03-12,2076,Tarjeta De Credito,csanchez,SES105459,Tel,TRM-512,Mañana,Camila H.,Mendoza
437,1115,2024-07-13,2074,Transferencia,lrodriguez,SES562458,Tel,TRM-893,Tarde,Valeria T.,Córdoba
438,1278,2022-03-22,2060,Cheque,csanchez,SES512731,Presencial,TRM-970,Mañana,Diego F.,Rosario


In [38]:
# Exportar el DataFrame de las cabeceras de órdenes
df_ordenes.to_csv('Processed/ventas_order_procesado.csv', index=False, encoding='utf-8')

print("✅ df_ordenes exportado exitosamente a Processed/ventas_order_procesado.csv")

✅ df_ordenes exportado exitosamente a Processed/ventas_order_procesado.csv
